In [1]:

import os

os.makedirs("data", exist_ok=True)

articles = [
    (1, "Telephone", "Electronique", 4500, 5),
    (2, "Casque", "Electronique", 350, 15),
    (3, "Clavier", "Informatique", 250, 0),
    (4, "Table", "Mobilier", 1800, 8),
    (5, "Chaise", "Mobilier", 700, 20),
    (6, "Stylo", "Papeterie", 10, 100),
    (7, "Imprimante", "Informatique", 2200, 4),
    (8, "Cahier", "Papeterie", 25, 50)
]

with open("data/articles.csv", "w", encoding="utf-8") as f:
    f.write("id_article,nom,categorie,prix,stock\n")
    for t in articles:
        f.write(",".join(str(x) for x in t) + "\n")

print(os.listdir("data"))


['.ipynb_checkpoints', 'articles.csv', 'produits.csv', 'ventes.csv']


In [5]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder.appName("GestionMagasin").master("local[*]").getOrCreate()
)

print("Session created")
sc = spark.sparkContext


Session created


In [6]:
rdd_lignes = sc.textFile("data/articles.csv")
rdd_lignes.collect()

['id_article,nom,categorie,prix,stock',
 '1,Telephone,Electronique,4500,5',
 '2,Casque,Electronique,350,15',
 '3,Clavier,Informatique,250,0',
 '4,Table,Mobilier,1800,8',
 '5,Chaise,Mobilier,700,20',
 '6,Stylo,Papeterie,10,100',
 '7,Imprimante,Informatique,2200,4',
 '8,Cahier,Papeterie,25,50']

In [9]:
rdd_articles = (
    rdd_lignes
    .filter(lambda l: not l.startswith("id_article"))
    .map(lambda l: l.split(","))
    .map(lambda l: (
        int(l[0]),
        l[1],
        l[2],
        int(l[3]),
        int(l[4])
    ))
)
rdd_articles.collect()

[(1, 'Telephone', 'Electronique', 4500, 5),
 (2, 'Casque', 'Electronique', 350, 15),
 (3, 'Clavier', 'Informatique', 250, 0),
 (4, 'Table', 'Mobilier', 1800, 8),
 (5, 'Chaise', 'Mobilier', 700, 20),
 (6, 'Stylo', 'Papeterie', 10, 100),
 (7, 'Imprimante', 'Informatique', 2200, 4),
 (8, 'Cahier', 'Papeterie', 25, 50)]

In [10]:
rdd_articles.take(3)

[(1, 'Telephone', 'Electronique', 4500, 5),
 (2, 'Casque', 'Electronique', 350, 15),
 (3, 'Clavier', 'Informatique', 250, 0)]

In [11]:
rdd_articles.filter(lambda x: x[4]< 10 ).map(lambda x: x[1]).collect()

['Telephone', 'Clavier', 'Table', 'Imprimante']

In [12]:
df_articles = spark.read.option("header",True).option("inferSchmea",True).csv("data/articles.csv")
df_articles.show()

+----------+----------+------------+----+-----+
|id_article|       nom|   categorie|prix|stock|
+----------+----------+------------+----+-----+
|         1| Telephone|Electronique|4500|    5|
|         2|    Casque|Electronique| 350|   15|
|         3|   Clavier|Informatique| 250|    0|
|         4|     Table|    Mobilier|1800|    8|
|         5|    Chaise|    Mobilier| 700|   20|
|         6|     Stylo|   Papeterie|  10|  100|
|         7|Imprimante|Informatique|2200|    4|
|         8|    Cahier|   Papeterie|  25|   50|
+----------+----------+------------+----+-----+



In [20]:
from pyspark.sql import functions as F
df_articles.filter((F.col("categorie")=="Informatique") & (F.col("prix") > 200)).select("nom","prix").orderBy(F.col("prix").asc()).show()

+----------+----+
|       nom|prix|
+----------+----+
|Imprimante|2200|
|   Clavier| 250|
+----------+----+



In [21]:
df_articles2 = df_articles.withColumn(
    "statu",
    F.when(F.col("stock") == 0 ,"rupture")
    .when(F.col("prix")<10 ,"faible")
    .otherwise("ok")
)
df_articles2.show()

+----------+----------+------------+----+-----+-------+
|id_article|       nom|   categorie|prix|stock|  statu|
+----------+----------+------------+----+-----+-------+
|         1| Telephone|Electronique|4500|    5|     ok|
|         2|    Casque|Electronique| 350|   15|     ok|
|         3|   Clavier|Informatique| 250|    0|rupture|
|         4|     Table|    Mobilier|1800|    8|     ok|
|         5|    Chaise|    Mobilier| 700|   20|     ok|
|         6|     Stylo|   Papeterie|  10|  100|     ok|
|         7|Imprimante|Informatique|2200|    4|     ok|
|         8|    Cahier|   Papeterie|  25|   50|     ok|
+----------+----------+------------+----+-----+-------+



In [24]:
df_articles.createOrReplaceTempView("articles")
query = spark.sql(""" select * from articles where categorie="Mobilier" """)
query.show()

+----------+------+---------+----+-----+
|id_article|   nom|categorie|prix|stock|
+----------+------+---------+----+-----+
|         4| Table| Mobilier|1800|    8|
|         5|Chaise| Mobilier| 700|   20|
+----------+------+---------+----+-----+



In [30]:
req = spark.sql(""" select count(*) as nbr_articles , max(prix) as max_prix from articles group by categorie having max(prix) > 500 order by max_prix desc """)
req.show()

+------------+--------+
|nbr_articles|max_prix|
+------------+--------+
|           2|     700|
|           2|    4500|
+------------+--------+

